In [4]:
#0.219

In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import KFold

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -------- Dataset：260次元特徴 + 相対速度 --------
class RelativeSpeedDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []

        print("\U0001F4E5 距離ファイル読み込み中...")
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue

            sid = fname.replace(".json", "")
            print(f"\U0001F4C2 処理中: {sid}")

            if sid not in self.distances:
                print(f"❌ スキップ: 距離情報なし")
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann.get("sequence", [])
            if len(seq) < 20:
                print(f"⚠️ スキップ: フレーム数 {len(seq)} 未満")
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)

            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)

            if len(dist) < 20:
                print(f"⚠️ スキップ: 距離データが20未満（{len(dist)}）")
                continue

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]

                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.concatenate([
                        d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                        f3[:20] * d1[:20], f11[:20] - f5[:20], np.abs(d1[:20])
                    ])
                except Exception as e:
                    print(f"❌ 特徴量結合エラー @ {sid} frame {i}: {e}")
                    continue

                if feat.shape[0] != 260:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid


# -------- 改良版モデル（Dropout + BatchNorm） --------
class ImprovedLinear260D(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(260, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)


# -------- クロスバリデーション学習ループ --------
def cross_validate_model(dataset, num_folds=5):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    all_losses = []

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
        print(f"\n=== Fold {fold+1}/{num_folds} ===")
        train_ds = torch.utils.data.Subset(dataset, train_idx)
        val_ds = torch.utils.data.Subset(dataset, val_idx)

        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ImprovedLinear260D().to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        criterion = nn.SmoothL1Loss()

        best_val_loss = float('inf')
        patience = 20
        counter = 0

        for epoch in range(100):
            model.train()
            total_train_loss = 0
            for feats, tgts, _ in tqdm(train_loader, desc=f"[Fold {fold+1} | Train {epoch+1}]"):
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item() * feats.size(0)

            model.eval()
            total_val_loss = 0
            with torch.no_grad():
                for feats, tgts, _ in val_loader:
                    feats, tgts = feats.to(device), tgts.to(device)
                    pred = model(feats)
                    loss = criterion(pred, tgts)
                    total_val_loss += loss.item() * feats.size(0)

            train_loss = total_train_loss / len(train_ds)
            val_loss = total_val_loss / len(val_ds)
            scheduler.step()

            print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f"model_fold{fold+1}.pth")
                print(f"✅ モデル保存: model_fold{fold+1}.pth（val_loss={val_loss:.4f}）")
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    print(f"🛑 Early stopping at epoch {epoch+1}")
                    break

        all_losses.append(best_val_loss)

    print("\n===== クロスバリデーション結果 =====")
    print("FoldごとのVal Loss:", all_losses)
    print("平均Val Loss:", np.mean(all_losses))


# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset260D(
        annot_root="./train_annotations",
        distance_json_path="../distance_ref_data.json",
        max_items=40000
    )

    print(f"✅ dataset loaded: {len(dataset)} samples")
    cross_validate_model(dataset, num_folds=5)


📥 距離ファイル読み込み中...
📂 処理中: 000
📂 処理中: 001
📂 処理中: 002
📂 処理中: 003
📂 処理中: 004
📂 処理中: 005
📂 処理中: 006
📂 処理中: 007
📂 処理中: 008
📂 処理中: 009
📂 処理中: 010
📂 処理中: 011
📂 処理中: 012
📂 処理中: 013
📂 処理中: 014
📂 処理中: 015
📂 処理中: 016
📂 処理中: 017
📂 処理中: 018
📂 処理中: 019
📂 処理中: 020
📂 処理中: 021
📂 処理中: 022
📂 処理中: 023
📂 処理中: 024
📂 処理中: 025
📂 処理中: 026
📂 処理中: 027
📂 処理中: 028
📂 処理中: 029
📂 処理中: 030
📂 処理中: 031
📂 処理中: 032
📂 処理中: 033
📂 処理中: 034
📂 処理中: 035
📂 処理中: 036
📂 処理中: 037
📂 処理中: 038
📂 処理中: 039
📂 処理中: 040
📂 処理中: 041
📂 処理中: 042
📂 処理中: 043
📂 処理中: 044
📂 処理中: 045
📂 処理中: 046
📂 処理中: 047
📂 処理中: 048
📂 処理中: 049
📂 処理中: 050
📂 処理中: 051
📂 処理中: 052
📂 処理中: 053
📂 処理中: 054
📂 処理中: 055
📂 処理中: 056
📂 処理中: 057
📂 処理中: 058
📂 処理中: 059
📂 処理中: 060
📂 処理中: 061
📂 処理中: 062
📂 処理中: 063
📂 処理中: 064
📂 処理中: 065
📂 処理中: 066
📂 処理中: 067
📂 処理中: 068
📂 処理中: 069
📂 処理中: 070
📂 処理中: 071
📂 処理中: 072
📂 処理中: 073
📂 処理中: 074
📂 処理中: 075
📂 処理中: 076
📂 処理中: 077
📂 処理中: 078
📂 処理中: 079
📂 処理中: 080
📂 処理中: 081
📂 処理中: 082
📂 処理中: 083
📂 処理中: 084
📂 処理中: 085
📂 処理中: 086
📂 処理中: 087
📂 処理中: 088
📂 処理

[Fold 1 | Train 1]: 100%|██████████| 500/500 [00:01<00:00, 291.57it/s]


Epoch 1 | Train Loss: 1.0467 | Val Loss: 0.5472
✅ モデル保存: model_fold1.pth（val_loss=0.5472）


[Fold 1 | Train 2]: 100%|██████████| 500/500 [00:01<00:00, 289.72it/s]


Epoch 2 | Train Loss: 0.7007 | Val Loss: 0.4702
✅ モデル保存: model_fold1.pth（val_loss=0.4702）


...

Epoch 48 | Train Loss: 0.4341 | Val Loss: 0.0455


[Fold 5 | Train 49]: 100%|██████████| 500/500 [00:01<00:00, 286.64it/s]


Epoch 49 | Train Loss: 0.4473 | Val Loss: 0.0565


[Fold 5 | Train 50]: 100%|██████████| 500/500 [00:01<00:00, 292.05it/s]


Epoch 50 | Train Loss: 0.4440 | Val Loss: 0.0810
🛑 Early stopping at epoch 50

===== クロスバリデーション結果 =====
FoldごとのVal Loss: [0.026020799715071916, 0.035559225246310236, 0.02504591067880392, 0.035006616815924645, 0.025278719928115607]
平均Val Loss: 0.029382254476845265


In [3]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- モデル（学習時と同じ構造） --------
class ImprovedLinear260D(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(260, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

# -------- 推論用 Dataset --------
class InferenceDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                feat = np.concatenate([
                    d, o, own_acc, d1, d2,
                    f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                    f3[:20] * d1[:20], f11[:20] - f5[:20], np.abs(d1[:20])
                ])

                if feat.shape[0] != 260:
                    continue

                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- 推論 + submission.json 作成 --------
def predict_with_cv_model(
    model_paths,
    annot_root,
    distance_json_path,
    save_path="submission_cv.json"
):
    dataset = InferenceDataset260D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = []
    for path in model_paths:
        model = ImprovedLinear260D().to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        models.append(model)

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            own_speeds = own_speeds.numpy()
            ensemble_preds = []
            for model in models:
                preds = model(feats).cpu().numpy()
                ensemble_preds.append(preds)
            avg_preds = np.mean(ensemble_preds, axis=0)
            abs_speeds = avg_preds + own_speeds

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存しました（scene数: {len(submission)}）")

# -------- 実行部 --------
if __name__ == "__main__":
    model_paths = [f"model_fold{i}.pth" for i in range(1, 6)]
    predict_with_cv_model(
        model_paths=model_paths,
        annot_root="../test2/test_annotations",
        distance_json_path="../testdistance/test_spline_smoothed_fixed.json",
        save_path="submission.json"
    )


100%|██████████| 395/395 [00:01<00:00, 226.94it/s]


✅ 完成: submission.json に保存しました（scene数: 239）
